In [ ]:
import numpy as np
import pandas as pd
import importlib

import src.utils.state as state
import src.utils.plottings as plotting
import src.utils.scenarioCalculator as scenarioCalculator
import src.utils.dataConverter as dataConverter
import src.utils.config as config
import src.utils.models as models
from src.utils.scenarios import Scenarios
from src.utils.models import Scenario

importlib.reload(plotting)
importlib.reload(state)
importlib.reload(scenarioCalculator)
importlib.reload(dataConverter)
importlib.reload(models)
importlib.reload(config)


# Preliminaries

## 1. Reading data

In [ ]:
df_cases = pd.read_excel("../resources/weekly_varicella_hun.xlsx", sheet_name="weekly_cases")
df_age_structured_cases = pd.read_excel("../resources/age_structured_vzv_hun.xlsx", sheet_name="varicella")
df_susceptibles = pd.read_excel("../resources/number_of_susceptibles_hun.xlsx", sheet_name="s0")
df_population = pd.read_excel("../resources/age_structured_population.xlsx", sheet_name="population")
df_births = pd.read_excel("../resources/births_hun.xlsx", sheet_name="births")
df_daily_births = pd.read_excel("../resources/births_hun.xlsx", sheet_name="daily", header=None)
df_deaths = pd.read_excel("../resources/age_structured_deaths_hun.xlsx", sheet_name="deaths")
df_death_proportions = pd.read_excel("../resources/age_structured_deaths_hun.xlsx", sheet_name="proportions")
df_vaccines = pd.read_excel("../resources/vaccine_coverage_hun.xlsx", sheet_name="vaccines")
df_contacts = pd.read_excel("../resources/contact_mtx_hun.xlsx", sheet_name="contacts", index_col=0)

## 2. Convert dates and set indices

### 2.1 Varicella data

#### 2.1.1 Weekly cases

In [ ]:
weekly_cases = dataConverter.convert_df_cases_to_weekly_cases(df_cases)
state.set_weekly_cases(weekly_cases)

# fill missing values
full_index = dataConverter.create_full_index_date_range(weekly_cases.index.max())
state.set_full_index(full_index)

state.set_weekly_cases_related_values()

#### 2.1.2 Age-structured data

In [ ]:
state.set_age_groups_and_nr_age_groups(df_age_structured_cases.iloc[:, 0].astype(str).tolist())
state.set_years(df_age_structured_cases.columns[1:].astype(int).tolist())

annual_cases = dataConverter.convert_df_age_structured_cases(df_age_structured_cases)
state.set_annual_cases(annual_cases)

In [ ]:
case_matrix = dataConverter.create_case_matrix(annual_cases, state.WEEKLY_CASES_FILLED, state.WEEKLY_CASES_FULL)
state.set_cases_matrix_and_global_min_max(case_matrix)

In [ ]:
state.set_age_structured_data_mask(state.WEEKLY_CASES_FILLED.index.year <= max(state.YEARS))
state.set_age_structured_data_index(state.WEEKLY_CASES_FILLED.loc[state.AGE_STRUCTURED_DATA_MASK].index)
i = case_matrix
state.set_nr_timesteps(len(state.WEEKLY_CASES_FILLED[state.AGE_STRUCTURED_DATA_MASK]))

### 2.2 Birth and death data; vaccination coverage

In [ ]:
state.set_weekly_index_related_values(state.WEEKLY_CASES_FILLED[state.AGE_STRUCTURED_DATA_MASK].index)

annual_deaths = dataConverter.convert_annual_deaths(df_deaths)
death_matrix = dataConverter.create_weekly_death_matrix(annual_deaths, state.WEEKLY_INDEX)
#death_matrix = dataConverter.create_weekly_death_matrix(annual_deaths, state.WEEKLY_CASES_FULL)
state.set_death_matrix(death_matrix)

#deaths_prop_matrix = dataConverter.convert_weekly_death_prop_data_to_death_prop_mtx(df_death_proportions)
#state.set_death_matrix(deaths_prop_matrix)

In [ ]:
#birth_mtx = dataConverter.convert_annual_birth_data_to_birth_mtx(df_births)
birth_mtx = dataConverter.convert_daily_birth_data_to_birth_mtx(df_daily_births)
state.set_birth_matrix(birth_mtx)
weekly_v1 = dataConverter.convert_annual_v1_data_to_weekly_v1(df_vaccines)
state.set_weekly_v1(weekly_v1)
weekly_v2 = dataConverter.convert_annual_v2_data_to_weekly_v2(df_vaccines)
state.set_weekly_v2(weekly_v2)

### 2.3 Initial values

#### 2.3.1 Number of susceptibles

In [ ]:
S0_vector = dataConverter.convert_data_to_s0(df_susceptibles)
state.set_s0_vector(S0_vector)

#### 2.3.2 Population size

In [ ]:
N0_vector = dataConverter.convert_data_to_n0(df_population)
state.set_n0_vector(N0_vector)

### 2.4 Contact matrix

In [ ]:
contacts0 = dataConverter.convert_data_to_contact_matrix(df_contacts)
state.set_contacts0(contacts0)

# Baseline scenario

## 3. Calculate number of susceptibles

In [ ]:
ALL_SCENARIOS: list[Scenario] = []

In [ ]:
state.set_weekly_v1_age_structured()
state.set_weekly_v2_age_structured()

scenarioCalculator.calculate_number_of_susceptible_cases_in_baseline(True)
ALL_SCENARIOS.append(state.BASELINE_SCENARIO)

## 4. Estimate R_t using renewal equation model

In [ ]:
scenarioCalculator.calculate_remainders(True)
plotting.plot_vector_with_index(state.RHO, state.AGE_STRUCTURED_DATA_INDEX, "Rho", "Date", "rho")

## 5. Plot results for R_t

In [ ]:
plotting.plot_remainders()

## 6. Plot results for the number of incidences

In [ ]:
scenarioCalculator.calculate_number_of_incidences_based_on_remainders()

plotting.plot_model_actual_scat_comparison(state.BASELINE_SCENARIO)
plotting.plot_annual_cases(state.BASELINE_SCENARIO)

In [ ]:
plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.V1, 0, 2, state.BASELINE_SCENARIO.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.I, 0, 4, state.BASELINE_SCENARIO.name)
plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.I, 5, 9, state.BASELINE_SCENARIO.name)
plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.I, 10, 12, state.BASELINE_SCENARIO.name)

In [ ]:
plotting.plot_scat_age_range_in_range(state.BASELINE_SCENARIO.I, state.AGE_STRUCTURED_DATA_INDEX, 0, 4, state.CASES_MATRIX, state.AGE_STRUCTURED_DATA_MASK)
plotting.plot_scat_age_range_in_range(state.BASELINE_SCENARIO.I, state.AGE_STRUCTURED_DATA_INDEX, 5, 9, state.CASES_MATRIX, state.AGE_STRUCTURED_DATA_MASK)
plotting.plot_scat_age_range_in_range(state.BASELINE_SCENARIO.I, state.AGE_STRUCTURED_DATA_INDEX, 10, 13, state.CASES_MATRIX, state.AGE_STRUCTURED_DATA_MASK)

# Change vaccination level

## 7. Incidences without vaccination

In [ ]:
scenario_no_vacc = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.NO_VACCINATION, 0, 0, True)
ALL_SCENARIOS.append(scenario_no_vacc)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_no_vacc)
plotting.plot_model_results_age_range_in_range(scenario_no_vacc.I, 0, 4, scenario_no_vacc.name)
plotting.plot_model_results_age_range_in_range(scenario_no_vacc.I, 5, 9, scenario_no_vacc.name)
plotting.plot_model_results_age_range_in_range(scenario_no_vacc.I, 10, 12, scenario_no_vacc.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_no_vacc.I, 0, 4, scenario_no_vacc.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_no_vacc.weekly_i_series, scenario_no_vacc.name)

## 8. Vaccination level = 0.75

In [ ]:
scenario_vacc_level_75: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.VACC_LEVEL_75, 0.75, 0, True)
ALL_SCENARIOS.append(scenario_vacc_level_75)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_vacc_level_75)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_75.I, 0, 4, scenario_vacc_level_75.name)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_75.I, 5, 9, scenario_vacc_level_75.name)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_75.I, 10, 12, scenario_vacc_level_75.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_vacc_level_75.weekly_i_series, scenario_vacc_level_75.name)

## 9. Vaccination level = 0.5

In [ ]:
scenario_vacc_level_50: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.VACC_LEVEL_50, 0.50, 0, True)
ALL_SCENARIOS.append(scenario_vacc_level_50)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_vacc_level_50)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_50.I, 0, 4, scenario_vacc_level_50.name)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_50.I, 5, 9, scenario_vacc_level_50.name)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_50.I, 10, 12, scenario_vacc_level_50.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_vacc_level_50.weekly_i_series, scenario_vacc_level_50.name)

## 10. Vaccination level = 0.25

In [ ]:
scenario_vacc_level_25: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.VACC_LEVEL_25, 0.25, 0, True)
ALL_SCENARIOS.append(scenario_vacc_level_25)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_vacc_level_25)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_25.I, 0, 4, scenario_vacc_level_25.name)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_25.I, 5, 9, scenario_vacc_level_25.name)
plotting.plot_model_results_age_range_in_range(scenario_vacc_level_25.I, 10, 12, scenario_vacc_level_25.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_vacc_level_25.weekly_i_series, scenario_vacc_level_25.name)

# Starting vaccination earlier

## 11. Start vaccination 1 year earlier (2018.09.01)

In [ ]:
scenario_starting_0_year_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.STARTING_1_YEAR_MINUS, 1, 0, True)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_starting_0_year_minus)
plotting.plot_model_results_age_range_in_range(scenario_starting_0_year_minus.I, 0, 4, scenario_starting_0_year_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_starting_0_year_minus.I, 5, 9, scenario_starting_0_year_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_starting_0_year_minus.I, 10, 12, scenario_starting_0_year_minus.name)

In [ ]:
scenario_1_year_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.STARTING_1_YEAR_MINUS, 1, -1, True)
ALL_SCENARIOS.append(scenario_1_year_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_1_year_minus)
plotting.plot_model_results_age_range_in_range(scenario_1_year_minus.I, 0, 4, scenario_1_year_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_1_year_minus.I, 5, 9, scenario_1_year_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_1_year_minus.I, 10, 12, scenario_1_year_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_1_year_minus.weekly_i_series, scenario_starting_0_year_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_1_year_minus.I, 0, 4, scenario_starting_0_year_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_1_year_minus.I, 5, 9, scenario_starting_0_year_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_1_year_minus.I, 10, 12, scenario_starting_0_year_minus.name)

## 12. Start vaccination 2 years earlier (2017.09.01)

In [ ]:
scenario_2_years_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.STARTING_2_YEAR_MINUS, 1, -2, True)
ALL_SCENARIOS.append(scenario_2_years_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_2_years_minus)
plotting.plot_model_results_age_range_in_range(scenario_2_years_minus.I, 0, 4, scenario_2_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_2_years_minus.I, 5, 9, scenario_2_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_2_years_minus.I, 10, 12, scenario_2_years_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(scenario_2_years_minus.V1, 0, 2, scenario_2_years_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_2_years_minus.weekly_i_series, scenario_2_years_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_2_years_minus.I, 0, 4, scenario_2_years_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_2_years_minus.I, 5, 9, scenario_2_years_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_2_years_minus.I, 10, 12, scenario_2_years_minus.name)

## 13. Start vaccination 3 years earlier (2016.09.01)

In [ ]:
scenario_3_years_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.STARTING_3_YEAR_MINUS, 1, -3, True)
ALL_SCENARIOS.append(scenario_3_years_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_3_years_minus)
plotting.plot_model_results_age_range_in_range(scenario_3_years_minus.I, 0, 4, scenario_3_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_3_years_minus.I, 5, 9, scenario_3_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_3_years_minus.I, 10, 12, scenario_3_years_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(scenario_3_years_minus.V1, 0, 2, scenario_3_years_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_3_years_minus.weekly_i_series, scenario_3_years_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_3_years_minus.I, 0, 4, scenario_3_years_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_3_years_minus.I, 5, 9, scenario_3_years_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_3_years_minus.I, 10, 12, scenario_3_years_minus.name)

## 14. Start vaccination 5 years earlier (2014.09.01)

In [ ]:
scenario_5_years_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.STARTING_5_YEAR_MINUS, 1, -5, True)
ALL_SCENARIOS.append(scenario_5_years_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_5_years_minus)
plotting.plot_model_results_age_range_in_range(scenario_5_years_minus.I, 0, 4, scenario_5_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_5_years_minus.I, 5, 9, scenario_5_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_5_years_minus.I, 10, 12, scenario_5_years_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(scenario_5_years_minus.V1, 1, 1, scenario_5_years_minus.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(scenario_5_years_minus.S, 0, 4, scenario_5_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_5_years_minus.S, 5, 9, scenario_5_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_5_years_minus.S, 10, 12, scenario_5_years_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_5_years_minus.weekly_i_series, scenario_5_years_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_5_years_minus.I, 0, 4, scenario_5_years_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_5_years_minus.I, 5, 9, scenario_5_years_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_5_years_minus.I, 10, 12, scenario_5_years_minus.name)

## 15. Starting the vaccination 8 years earlier (2011.09.01)

In [ ]:
scenario_8_years_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.STARTING_8_YEAR_MINUS, 1, -8, True)
ALL_SCENARIOS.append(scenario_8_years_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_8_years_minus)
plotting.plot_model_results_age_range_in_range(scenario_8_years_minus.I, 0, 4, scenario_8_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_8_years_minus.I, 5, 9, scenario_8_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_8_years_minus.I, 10, 12, scenario_8_years_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_8_years_minus.weekly_i_series, scenario_8_years_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_8_years_minus.I, 0, 4, scenario_8_years_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_8_years_minus.I, 5, 9, scenario_8_years_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_8_years_minus.I, 10, 12, scenario_8_years_minus.name)

## 16. Starting the vaccination 13 years earlier (2006.09.01)

In [ ]:
scenario_13_years_minus: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.STARTING_13_YEAR_MINUS, 1, -13, True)
ALL_SCENARIOS.append(scenario_13_years_minus)

In [ ]:
plotting.plot_model_actual_scat_comparison(scenario_13_years_minus)
plotting.plot_model_results_age_range_in_range(scenario_13_years_minus.I, 0, 4, scenario_13_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_13_years_minus.I, 5, 9, scenario_13_years_minus.name)
plotting.plot_model_results_age_range_in_range(scenario_13_years_minus.I, 10, 12, scenario_13_years_minus.name)

In [ ]:
plotting.plot_compare_scenario_to_actual_case(scenario_13_years_minus.weekly_i_series, scenario_13_years_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_13_years_minus.I, 0, 4, scenario_13_years_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_13_years_minus.I, 5, 9, scenario_13_years_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(scenario_13_years_minus.I, 10, 12, scenario_13_years_minus.name)

In [ ]:
plotting.plot_cumulative_i_and_v(scenario_13_years_minus)

# Summaries

## 17. Cumulative plots

In [ ]:
mask = np.ones(len(state.AGE_STRUCTURED_DATA_INDEX), dtype=bool)
plotting.plot_cumulative_cases(ALL_SCENARIOS, state.AGE_STRUCTURED_DATA_INDEX, mask)

In [ ]:
index_after_2016_1_1 = state.WEEKLY_INDEX[state.AFTER_2016_1_1_MASK]
plotting.plot_cumulative_cases(ALL_SCENARIOS, index_after_2016_1_1, state.AFTER_2016_1_1_MASK)

## 18. Heatmaps

In [ ]:
for scenario in ALL_SCENARIOS:
    df_scenario = pd.DataFrame(scenario.I, index=state.AGE_GROUPS, columns=state.WEEKLY_INDEX)
    plotting.plot_heatmap(df_scenario, "The change of varicella age distribution - " + scenario.name)

# Without Covid

In [ ]:
# df_remainders = pd.DataFrame(state.REMAINDERS[:,3:].T, index=state.WEEKLY_INDEX[3:], columns=state.AGE_GROUPS)
# timerange_for_mean = (df_remainders.index > config.GlobalConfig.START_OF_VACCINATION + pd.DateOffset(years=-5)) & (df_remainders.index < config.GlobalConfig.START_OF_VACCINATION)
# remainders_mean: pd.DataFrame = (df_remainders[timerange_for_mean].groupby(df_remainders.index[timerange_for_mean]
#                                           .map(lambda index: index.week)).mean())
# remainders_series: pd.Series = pd.Series(np.sum(state.REMAINDERS, axis=0), index=state.WEEKLY_INDEX)
# no_covid_remainders: np.ndarray = np.zeros((len(state.AGE_GROUPS), len(state.WEEKLY_INDEX)))
# for t in range(0, state.NR_TIMESTEPS):
#     if (state.WEEKLY_INDEX[t] < config.GlobalConfig.START_OF_RESTRICTIONS) or (state.WEEKLY_INDEX[t] > pd.Timestamp("2020-09-01")):
#         no_covid_remainders[:,t] = state.REMAINDERS[:,t]
#     else:
#         week_of_year: int = state.WEEKLY_INDEX[t].week
#         no_covid_remainders[:,t] = remainders_mean.loc[week_of_year]
# state.set_no_covid_remainders(no_covid_remainders)

## 14. Use mean R_t

In [ ]:
def calculate_mean_during_covid(data: np.ndarray[float], index: pd.Series) -> np.ndarray[float]:
    df: pd.DataFrame = pd.DataFrame(data.T, index=index, columns=state.AGE_GROUPS)
    timerange_for_mean = ((df.index > config.GlobalConfig.START_OF_VACCINATION + pd.DateOffset(years=-14))
                         & (df.index < config.GlobalConfig.START_OF_VACCINATION))
    #timerange_for_mean = ((df.index > pd.Timestamp("2020-03-16") + pd.DateOffset(years=-13))
    #                     & (df.index < pd.Timestamp("2020-03-16")))
    mean_data: pd.DataFrame = df[timerange_for_mean].groupby(df.index[timerange_for_mean].map(lambda i: i.week)).mean()
    no_cov_data: np.ndarray = np.zeros((len(state.AGE_GROUPS), len(state.WEEKLY_INDEX)))
    #no_cov_data: np.ndarray = np.zeros((len(state.AGE_GROUPS), len(state.WEEKLY_INDEX)))
    min_t = len(state.WEEKLY_INDEX) - len(index)
    max_t = len(state.WEEKLY_INDEX)
    for t in range(min_t, max_t):
        no_cov_data[:, t] = data[:, t - min_t]
        if (state.WEEKLY_INDEX[t] > pd.Timestamp("2020-03-16")) and (state.WEEKLY_INDEX[t] < pd.Timestamp("2025-01-01")):
            #timerange_for_mean = ((df.index > pd.Timestamp("2014-09-01"))
            #                      & (df.index <state.WEEKLY_INDEX[t]))
            #mean_data: pd.DataFrame = df[timerange_for_mean].groupby(df.index[timerange_for_mean].map(lambda i: i.week)).mean()
            week_of_year: int = state.WEEKLY_INDEX[t].week
            no_cov_data[:, t] = mean_data.loc[week_of_year][:]
            #no_cov_data[2:, t] = mean_data.loc[week_of_year][2:]
    return no_cov_data

In [ ]:
mean_remainders: np.ndarray[float] = calculate_mean_during_covid(state.REMAINDERS[:,3:], state.WEEKLY_INDEX[3:])
state.set_no_covid_remainders(mean_remainders)

In [ ]:
mean_remainders: np.ndarray[float] = calculate_mean_during_covid(state.REMAINDERS[:,3:], state.WEEKLY_INDEX[3:])
state.set_no_covid_remainders(mean_remainders)
mean_births: np.ndarray[float] = calculate_mean_during_covid(state.BIRTH_MATRIX, state.WEEKLY_INDEX)
state.set_no_covid_births(mean_births)
#state.set_no_covid_births(state.BIRTH_MATRIX)
mean_deaths: np.ndarray[float] = calculate_mean_during_covid(state.DEATHS_MATRIX, state.WEEKLY_INDEX)
#state.set_no_covid_deaths(state.DEATHS_MATRIX)
state.set_no_covid_deaths(mean_deaths)

In [ ]:
# scenario_no_cov: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.NO_COVID, 1, 0, False)
# ALL_SCENARIOS.append(scenario_no_cov)

In [ ]:
# plotting.plot_model_actual_scat_comparison(scenario_no_cov)
# plotting.plot_model_results_age_range_in_range(scenario_no_cov.I, 0, 4, scenario_no_cov.name)
# plotting.plot_model_results_age_range_in_range(scenario_no_cov.I, 5, 9, scenario_no_cov.name)
# plotting.plot_model_results_age_range_in_range(scenario_no_cov.I, 10, 12, scenario_no_cov.name)

In [ ]:
# plotting.plot_model_results_age_range_in_range(scenario_no_cov.S, 0, 4, scenario_no_cov.name)
# plotting.plot_model_results_age_range_in_range(scenario_no_cov.S, 5, 9, scenario_no_cov.name)
# plotting.plot_model_results_age_range_in_range(scenario_no_cov.S, 10, 12, scenario_no_cov.name)

In [ ]:
# scenario_no_cov_no_vacc: Scenario = scenarioCalculator.calculate_number_of_cases_for_scenario(Scenarios.NO_COV_NO_VACC, 0, 0, False)
# plotting.plot_model_actual_scat_comparison(scenario_no_cov_no_vacc)
# plotting.plot_model_results_age_range_in_range(scenario_no_cov_no_vacc.I, 0, 4, Scenarios.NO_COV_NO_VACC)
# plotting.plot_model_results_age_range_in_range(scenario_no_cov_no_vacc.I, 5, 9, Scenarios.NO_COV_NO_VACC)
# plotting.plot_model_results_age_range_in_range(scenario_no_cov_no_vacc.I, 10, 12, Scenarios.NO_COV_NO_VACC)

In [ ]:
# plotting.plot_model_results_age_range_in_range(scenario_no_cov_no_vacc.S, 0, 4, Scenarios.NO_COV_NO_VACC)
# plotting.plot_model_results_age_range_in_range(scenario_no_cov_no_vacc.S, 5, 9, Scenarios.NO_COV_NO_VACC)

In [ ]:
# test_scen_no_cov = scenarioCalculator.init_scenario("test_scen_no_cov")
# mean_s = calculate_mean_during_covid(state.BASELINE_SCENARIO.S, state.WEEKLY_INDEX)
# test_scen_no_cov.S = mean_s
#
# A = state.NR_AGE_GROUPS
# T = state.NR_TIMESTEPS
# i_model = np.zeros((A, T))
# for t in range(0, config.GlobalConfig.MAX_TAU):
#     i_model[:, t] = state.CASES_MATRIX[:, t]
#
# for t in range(config.GlobalConfig.MAX_TAU, T - 1):
#     s_t = test_scen_no_cov.S[:,t] / state.POPULATION[:, t]
#     #r_t_scen = state.REMAINDERS[:, t] * s_t
#     r_t_scen = state.NO_COVID_REMAINDERS[:, t] * s_t
#     denominator = scenarioCalculator.calculate_denominator(i_model, t)
#     i_model[:, t] = r_t_scen * denominator
#
# test_scen_no_cov.I = i_model
# test_scen_no_cov.compute_i_series()

In [ ]:
# plotting.plot_model_actual_scat_comparison(test_scen_no_cov)
# plotting.plot_model_results_age_range_in_range(test_scen_no_cov.I, 0, 4, "test_scen_no_cov")
# plotting.plot_model_results_age_range_in_range(test_scen_no_cov.I, 5, 9, "test_scen_no_cov")
# plotting.plot_model_results_age_range_in_range(test_scen_no_cov.I, 10, 12, "test_scen_no_cov")
# plotting.plot_compare_scenario_to_actual_case(test_scen_no_cov.weekly_i_series, "test_scen_no_cov")

In [ ]:
# plotting.plot_model_results_age_range_in_range(test_scen_no_cov.S, 0, 4, "test_scen_no_cov")
# plotting.plot_model_results_age_range_in_range(test_scen_no_cov.S, 5, 9, "test_scen_no_cov")

In [ ]:
# plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.S, 0, 4, state.BASELINE_SCENARIO.name)
# plotting.plot_model_results_age_range_in_range(state.BASELINE_SCENARIO.S, 5, 9, state.BASELINE_SCENARIO.name)

## test mean_i

In [ ]:
mean_i = calculate_mean_during_covid(state.BASELINE_SCENARIO.I, state.WEEKLY_INDEX)
state.set_mean_i(mean_i)
scenarioCalculator.calculate_number_of_susceptible_cases_in_baseline(False)
state.MEAN_I_SCENARIO.I = mean_i
state.MEAN_I_SCENARIO.compute_i_series()

In [ ]:
plotting.plot_model_results_age_range_in_range(state.MEAN_I_SCENARIO.S, 0, 4, "no covid baseline")
plotting.plot_model_results_age_range_in_range(state.MEAN_I_SCENARIO.S, 5, 9, "no covid baseline")
plotting.plot_model_results_age_range_in_range(state.MEAN_I_SCENARIO.S, 10, 12, "no covid baseline")

In [ ]:
plotting.plot_model_results_age_range_in_range(state.MEAN_I_SCENARIO.I, 0, 4, "no covid baseline")
plotting.plot_model_results_age_range_in_range(state.MEAN_I_SCENARIO.I, 5, 9, "no covid baseline")
plotting.plot_model_results_age_range_in_range(state.MEAN_I_SCENARIO.I, 10, 12, "no covid baseline")

In [ ]:
scenarioCalculator.calculate_remainders(False)

In [ ]:
plotting.plot_model_results_age_range_in_range(state.MEAN_REMAINDERS, 0, 4, "mean remainders")
plotting.plot_model_results_age_range_in_range(state.MEAN_REMAINDERS, 5, 9, "mean remainders")
plotting.plot_model_results_age_range_in_range(state.MEAN_REMAINDERS, 10, 12, "mean remainders")

In [ ]:
test = scenarioCalculator.calculate_number_of_cases_for_scenario("test", 0, 0, False)

In [ ]:
plotting.plot_model_actual_scat_comparison(test)
plotting.plot_model_results_age_range_in_range(test.I, 0, 4, "test_scen_no_cov")
plotting.plot_model_results_age_range_in_range(test.I, 5, 9, "test_scen_no_cov")
plotting.plot_model_results_age_range_in_range(test.I, 10, 12, "test_scen_no_cov")

In [ ]:
test_vacc = scenarioCalculator.calculate_number_of_cases_for_scenario("test", 1, 0, False)

In [ ]:
plotting.plot_model_actual_scat_comparison(test_vacc)
plotting.plot_model_results_age_range_in_range(test_vacc.I, 0, 4, "test_scen_no_cov")
plotting.plot_model_results_age_range_in_range(test_vacc.I, 5, 9, "test_scen_no_cov")
plotting.plot_model_results_age_range_in_range(test_vacc.I, 10, 12, "test_scen_no_cov")

In [ ]:
plotting.plot_compare_scenario_to_actual_case(test_vacc.weekly_i_series, test_vacc.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(test_vacc.I, 0, 4, test_vacc.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(test_vacc.I, 5, 9, test_vacc.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(test_vacc.I, 10, 12, test_vacc.name)

In [ ]:
plotting.plot_model_results_age_range_in_range(test_vacc.S, 0, 4, "test_scen_no_cov")
plotting.plot_model_results_age_range_in_range(test_vacc.S, 5, 9, "test_scen_no_cov")
plotting.plot_model_results_age_range_in_range(test_vacc.S, 10, 12, "test_scen_no_cov")

In [ ]:
test_vacc_5_year_minus = scenarioCalculator.calculate_number_of_cases_for_scenario("test", 1, -5, False)

In [ ]:
plotting.plot_model_actual_scat_comparison(test_vacc_5_year_minus)
plotting.plot_model_results_age_range_in_range(test_vacc_5_year_minus.I, 0, 4, "test_scen_no_cov")
plotting.plot_model_results_age_range_in_range(test_vacc_5_year_minus.I, 5, 9, "test_scen_no_cov")
plotting.plot_model_results_age_range_in_range(test_vacc_5_year_minus.I, 10, 12, "test_scen_no_cov")

In [ ]:
plotting.plot_model_results_age_range_in_range(test_vacc_5_year_minus.S, 0, 4, "test_scen_no_cov")
plotting.plot_model_results_age_range_in_range(test_vacc_5_year_minus.S, 5, 9, "test_scen_no_cov")
plotting.plot_model_results_age_range_in_range(test_vacc_5_year_minus.S, 10, 12, "test_scen_no_cov")

In [ ]:
plotting.plot_model_results_age_range_in_range(test_vacc_5_year_minus.V1, 0, 2, "test_scen_no_cov")

In [ ]:
plotting.plot_compare_scenario_to_actual_case(test_vacc_5_year_minus.weekly_i_series, test_vacc_5_year_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(test_vacc_5_year_minus.I, 0, 4, test_vacc_5_year_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(test_vacc_5_year_minus.I, 5, 9, test_vacc_5_year_minus.name)
plotting.plot_compare_scenario_to_actual_case_for_age_groups(test_vacc_5_year_minus.I, 10, 12, test_vacc_5_year_minus.name)